# 01 — Aumentación del Dataset EU AI Act

Este notebook expande el dataset original de ~600 ejemplos usando:
- **Paráfrasis LLM** (AWS Bedrock Nova Lite): 3 variaciones por ejemplo
- **Back-translation** (ES → EN → ES via Google Translate): 1 variación por ejemplo

Resultado esperado: ~2400–3000 ejemplos en `data/dataset_augmented.jsonl`

> ⚠️ Este notebook hace llamadas a Bedrock (1 llamada por ejemplo). Usa `LIMIT` para probar antes de ejecutar completo.

In [1]:
import json
import sys
import logging
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# Añadir raíz del proyecto al path
ROOT = Path().resolve().parents[3]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.WARNING)
print('ROOT:', ROOT)

ROOT: C:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_CSV = ROOT / 'src/classifier/classifier_dataset_fusionado/datasets/eu_ai_act_flagged_es_limpio.csv'
OUTPUT_JSONL = ROOT / 'src/classifier/bert_pipeline/data/dataset_augmented.jsonl'
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

# --- CONFIGURACIÓN ---
N_PARAPHRASES = 3     # paráfrasis LLM por ejemplo
BACK_TRANSLATION = True
LIMIT = None         # None = dataset completo | int = modo debug (ej: 10)

print(f'Dataset origen : {DATASET_CSV}')
print(f'Output JSONL   : {OUTPUT_JSONL}')
print(f'N_PARAPHRASES  : {N_PARAPHRASES}')
print(f'BACK_TRANSLATION: {BACK_TRANSLATION}')
print(f'LIMIT          : {LIMIT}')

Dataset origen : C:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final\src\classifier\classifier_dataset_fusionado\datasets\eu_ai_act_flagged_es_limpio.csv
Output JSONL   : C:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final\src\classifier\bert_pipeline\data\dataset_augmented.jsonl
N_PARAPHRASES  : 3
BACK_TRANSLATION: True
LIMIT          : None


## 1. Cargar dataset original

In [3]:
df = pd.read_csv(DATASET_CSV)[['descripcion', 'etiqueta_normalizada']].dropna()
if LIMIT:
    df = df.head(LIMIT)

print(f'Ejemplos cargados: {len(df)}')
print('\nDistribución de clases:')
print(df['etiqueta_normalizada'].value_counts().to_string())
df.head(3)

Ejemplos cargados: 832

Distribución de clases:
etiqueta_normalizada
riesgo_minimo      208
riesgo_limitado    208
alto_riesgo        208
inaceptable        208


,descripcion,etiqueta_normalizada
0,Una ciudad inteligente implementa un sistema d...,riesgo_minimo
1,Un chatbot de servicio al cliente utiliza el r...,riesgo_limitado
2,Un sistema de inteligencia artificial utilizad...,alto_riesgo


## 2. Paráfrasis via LLM (Bedrock Nova Lite)

Probamos con un ejemplo antes de ejecutar el bucle completo.

In [4]:
from src.classifier.bert_pipeline.augmentation.paraphrase_llm import paraphrase

ejemplo = df['descripcion'].iloc[0]
etiqueta = df['etiqueta_normalizada'].iloc[0]
print(f'Original ({etiqueta}):\n{ejemplo}\n')

parafraseos = paraphrase(ejemplo, n=N_PARAPHRASES, label=etiqueta)
for i, p in enumerate(parafraseos, 1):
    print(f'Paráfrasis {i}: {p}')

Original (riesgo_minimo):
Una ciudad inteligente implementa un sistema de inteligencia artificial para la optimización del semáforo que incluye un panel público que explica el propósito del sistema, las fuentes de datos y la lógica de toma de decisiones, con actualizaciones en tiempo real sobre cómo los ajustes de la inteligencia artificial impactan el flujo de tráfico.

Paráfrasis 1: En una metrópolis tecnológica, se ha puesto en marcha un sistema de inteligencia artificial para la gestión de semáforos que presenta un monitor público explicando su objetivo, las fuentes de información y el proceso de decisión, ofreciendo informes continuos sobre cómo las modificaciones de la inteligencia artificial afectan el tráfico.
Paráfrasis 2: Un municipio avanzado ha adoptado un sistema de IA para optimizar la sincronización de los semáforos, con un tablero abierto que detalla la finalidad del sistema, las bases de datos utilizadas y la metodología de las decisiones, y con avisos actualizados sob

## 3. Back-translation (ES → EN → ES)

In [5]:
from src.classifier.bert_pipeline.augmentation.back_translation import back_translate

bt = back_translate(ejemplo)
print(f'Original     : {ejemplo}')
print(f'Back-transl. : {bt}')

Original     : Una ciudad inteligente implementa un sistema de inteligencia artificial para la optimización del semáforo que incluye un panel público que explica el propósito del sistema, las fuentes de datos y la lógica de toma de decisiones, con actualizaciones en tiempo real sobre cómo los ajustes de la inteligencia artificial impactan el flujo de tráfico.
Back-transl. : 


## 4. Bucle de aumentación completo

El checkpoint se guarda línea a línea (JSONL), por lo que si el proceso se interrumpe
se puede reanudar filtrando los ejemplos ya procesados.

In [ ]:
import random
random.seed(42)

total = 0
errors = 0

with OUTPUT_JSONL.open('w', encoding='utf-8') as out:
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Aumentando'):
        texto = str(row['descripcion'])
        etiqueta = str(row['etiqueta_normalizada'])

        def write(desc, source):
            global total
            out.write(json.dumps({'descripcion': desc, 'etiqueta': etiqueta, 'source': source}, ensure_ascii=False) + '\n')
            total += 1

        # Original
        write(texto, 'original')

        # Paráfrasis LLM
        try:
            for i, p in enumerate(paraphrase(texto, n=N_PARAPHRASES, label=etiqueta), 1):
                write(p, f'paraphrase_{i}')
        except Exception:
            errors += 1

        # Back-translation
        # None = error en el traductor (distinguible de "" = sin variación léxica)
        if BACK_TRANSLATION:
            bt = back_translate(texto)
            if bt is None:
                errors += 1  # error real, logeado internamente por back_translate
            elif bt:
                write(bt, 'back_translation')  # "" = sin variación, se omite sin contar error

print(f'\nTotal ejemplos generados : {total}')
print(f'Factor de aumento        : x{total / len(df):.1f}')
print(f'Errores                  : {errors}')
print(f'Guardado en              : {OUTPUT_JSONL}')

Aumentando:   7%|▋         | 61/832 [02:06<40:32,  3.15s/it]

## 5. Verificación del resultado

In [ ]:
records = [json.loads(line) for line in OUTPUT_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
df_aug = pd.DataFrame(records)

print(f'Total filas en JSONL: {len(df_aug)}')
print('\nDistribución de fuentes:')
print(df_aug['source'].value_counts().to_string())
print('\nDistribución de clases (aumentado):')
print(df_aug['etiqueta'].value_counts().to_string())
df_aug.head()